In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
train_ds = datasets.CIFAR10(root="/tmp/data", train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(root="/tmp/data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [179]:
import einops

def patch_embedding_conv(x, proj_W, proj_b, P, D):
    B, C, H, W = x.size()
    assert H % P ==0 and W % P == 0, "H, W not divisible by P"
    N = H * W // P**2
    conv = nn.Conv2d(in_channels=C, out_channels=D, kernel_size=P, stride=P)
    conv._parameters['weight'] = proj_W.view(D, C, P, P)
    conv._parameters['bias'] = proj_b
    out = conv(x) # (B, D, H//P, W//P)
    return out.reshape(B, D, -1).transpose(-1, -2)
    
def patch_embedding_manual(x, proj_W, proj_b, P, D):
    B, C, H, W = x.size()
    assert H % P ==0 and W % P == 0, "H, W not divisible by P"
    N = H * W // P**2
    out = []
    for i in range(0, H, P):
        for j in range(0, W, P):
            out.append(
                x[:, :, i:i+P, j:j+P].reshape(B, -1) @ proj_W.T + proj_b
            )
    return torch.stack(out, dim=1)

def patch_embedding_einops(x, proj_W, proj_b, P, D):
    patches = einops.rearrange(x, 'b c (h p1) (w p2) -> b (h w) (c p1 p2)', p1=P, p2=P) # B, N, C*P*P
    print(patches.shape)
    return patches @ proj_W.T + proj_b

def patch_embedding_torch(x, proj_W, proj_b, P, D):
    B, C, H, W = x.shape
    N = H * W // P**2
    patches = img.reshape(B, C, H//P, P, W//P, P).permute(0, 2, 4, 1, 3, 5).reshape(B, N, C*P*P)
    return patches @ proj_W.T + proj_b

B, C, H, W, P, D = 64, 3, 32, 32, 4, 16
img = torch.randn(B, C, H, W)
proj_W = torch.randn(D, C * P * P)
proj_b = torch.randn(D)
pte_conv = patch_embedding_conv(img, proj_W, proj_b, P, D)
pte_man = patch_embedding_manual(img, proj_W, proj_b, P, D)
pte_ein = patch_embedding_einops(img, proj_W, proj_b, P, D)
pte_tor = patch_embedding_torch(img, proj_W, proj_b, P, D)
print((pte_conv - pte_man).abs().max().item())
print((pte_ein - pte_man).abs().max().item())
print((pte_tor - pte_man).abs().max().item())

torch.Size([64, 64, 48])
9.5367431640625e-06
0.0
0.0


In [190]:
class LayerNorm(nn.Module):
    VARIANCE_UNBIASED = True # False will make it identical to nn.LayerNorm
    def __init__(self, norm_shape, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.norm_ndim = 1 if type(norm_shape) is int else len(norm_shape)
        self.gamma = nn.Parameter(torch.ones(norm_shape))
        self.beta = nn.Parameter(torch.zeros(norm_shape)) 

    def forward(self, x):
        mean = torch.mean(x, dim=list(range(-self.norm_ndim, 0)), keepdim=True) # (*, *norm_shape) 
        var = torch.var(x, dim=list(range(-self.norm_ndim, 0)), keepdim=True, unbiased=self.VARIANCE_UNBIASED) # (*, *norm_shape)
        x = (x - mean) / torch.sqrt(var + self.eps)
        x = self.gamma * x - self.beta # x -> (*, *norm_shape); self.gamma -> (*norm_shape) will be broadcasted
        return x
    
class PatchEmbeddingConv(nn.Module):
    def __init__(self, P, C, D):
        super().__init__()
        self.n_embd = D
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels=C, out_channels=D, kernel_size=P, stride=P),
            nn.LayerNorm(D),
        )
        
    def forward(self, x): # B, C, H, W
        out = self.conv(x) # (B, D, H//P, W//P)
        return out.reshape(x.shape[0], self.n_embd, -1).transpose(-1, -2) # (B, N, D)
    

In [ ]:
torch.manual_seed(37)

class PatchEmbedding(nn.Module):
    def __init__(self, P, C, D):
        super().__init__()
        self.proj = nn.Sequential(
            LayerNorm(C*P*P),
            nn.Linear(in_features=C*P*P, out_features=D),
            LayerNorm(D),
        )
        
    def forward(self, x): # B, C, H, W
        x = einops.rearrange(x, 'b c (h p1) (w p2) -> b (h w) (c p1 p2)', p1=P, p2=P) # B, N, C*P*P
        return self.proj(x)

class FeedForward(nn.Module):
    BLOAT = 4
    def __init__(self, D):
        super().__init__()
        self.net = nn.Sequential(
            LayerNorm(D),
            nn.Linear(D, self.BLOAT * D),
            nn.GELU(),
            nn.Linear(self.BLOAT * D, D),
        )
        self.ln = LayerNorm(D)
        
    def forward(self, x):
        return x + self.net(self.ln(x))

class Attention(nn.Module):
    def __init__(self, D, n_heads):
        super().__init__()
        self.msa = nn.MultiheadAttention(embed_dim=D, num_heads=n_heads, batch_first=True)
        self.ln = LayerNorm(D)
        
    def forward(self, x):
        x = self.ln(x)
        return x + self.msa(x, x, x)[0]

class Encoder(nn.Module):
    def __init__(self, D, n_heads):
        super().__init__()
        self.attn = Attention(D, n_heads)
        self.ffwd = FeedForward(D)

    def forward(self, x):
        return self.ffwd(self.attn(x))

class ViT(nn.Module):
    @staticmethod
    def positional_encoding(N, D, freq_constant=10):
        pos_enc = torch.empty(N, D)
        wave_scaler = freq_constant ** (torch.arange(0, D, 2) / D)
        pos = torch.arange(1, N + 1)
        pos_enc[:, ::2] = torch.sin(pos.unsqueeze(1) / wave_scaler[:(D - 1)//2 + 1].unsqueeze(0))
        pos_enc[:, 1::2] = torch.cos(pos.unsqueeze(1) / wave_scaler[:D//2].unsqueeze(0))
        return pos_enc

    def __init__(self, H, W, C, P, D, n_heads, L, n_classes):
        super().__init__()
        assert H % P == 0 and W % P == 0, "P must be a factor of H and W"
        N = H * W // P**2 # Add the classification patch
        self.n_embd = D
        # self.register_buffer('pos_enc', ViT.positional_encoding(N, D))
        self.pos_embd = nn.Parameter(torch.randn(N + 1, D))
        self.patch_embd = PatchEmbedding(P, C, D)
        self.encoders = nn.Sequential(*[Encoder(D, n_heads) for _ in range(L)])
        self.cls_patch_embd = nn.Parameter(torch.randn(D))
        self.cls_ln = LayerNorm(D)
        self.classifier = nn.Linear(D, n_classes)

    def forward(self, x): # B, C, H, W
        img_patches = self.patch_embd(x)
        out = torch.cat([ self.cls_patch_embd.broadcast_to(x.size(0), 1, self.n_embd), img_patches ], dim=1) # (B, seq_len = N + 1, D)
        out = out + self.pos_embd
        out = self.encoders(out)
        out = self.classifier(self.cls_ln(out[:, 0, :].squeeze(1)))
        return out

In [224]:
def eval_model(model, loader, device, criterion):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            loss_sum += loss.item() * y.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

def train(model, epochs, device):
    # Model, loss, optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-2)
    lr_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=10, eta_min=1e-6)

    # Train
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            running_loss += loss.item() * y.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        test_loss, test_acc = eval_model(model, test_loader, device, criterion)
        print(f"Epoch {epoch}: train_loss={train_loss:.4f} | test_loss={test_loss:.4f} | test_acc={test_acc:.4f}")

def count_params(model):
    count = 0
    for n, p in model.named_parameters():
        # print(n, p.shape)
        count += p.numel()
    return count

In [228]:
device = 'cuda'
model = ViT(H=32, W=32, C=3, P=4, D=4, n_heads=2, L=1, n_classes=10).to(device)

In [229]:
count_params(model)

874

In [233]:
for n, p in model.encoders[0].named_parameters():
    print(n, p.numel())

attn.msa.in_proj_weight 48
attn.msa.in_proj_bias 12
attn.msa.out_proj.weight 16
attn.msa.out_proj.bias 4
attn.ln.gamma 4
attn.ln.beta 4
ffwd.net.0.gamma 4
ffwd.net.0.beta 4
ffwd.net.1.weight 64
ffwd.net.1.bias 16
ffwd.net.3.weight 64
ffwd.net.3.bias 4
ffwd.ln.gamma 4
ffwd.ln.beta 4


In [219]:
train(model, 10,'cuda')

Epoch 1: train_loss=1.3149 | test_loss=1.4202 | test_acc=0.5822
Epoch 2: train_loss=1.3071 | test_loss=1.4676 | test_acc=0.5657
Epoch 3: train_loss=1.2997 | test_loss=1.4362 | test_acc=0.5817
Epoch 4: train_loss=1.2840 | test_loss=1.4647 | test_acc=0.5669
Epoch 5: train_loss=1.2926 | test_loss=1.4300 | test_acc=0.5817
Epoch 6: train_loss=1.2811 | test_loss=1.4107 | test_acc=0.5931
Epoch 7: train_loss=1.2801 | test_loss=1.4541 | test_acc=0.5633
Epoch 8: train_loss=1.2615 | test_loss=1.4655 | test_acc=0.5580
Epoch 9: train_loss=1.2646 | test_loss=1.4688 | test_acc=0.5585
Epoch 10: train_loss=1.2573 | test_loss=1.4399 | test_acc=0.5817
